In [ ]:
from pathlib import Path

import pymupdf

# Run from the project folder containing the PDFs and tessdata/.
TESSDATA_DIR = Path("tessdata").resolve()

def inspect_page(page):
    # 1. Count native words
    words = page.get_text("words")

    # 2. Find raster images displayed on the page
    images = page.get_image_info()

    # 3. Measure the largest image's page coverage
    page_area = page.rect.get_area()

    largest_coverage = max(
        (
            (pymupdf.Rect(img["bbox"]) & page.rect).get_area()
            / page_area
            for img in images
        ),
        default=0.0
    )

    return {
        "native_words": len(words),
        "image_count": len(images),
        "largest_image_fraction": round(largest_coverage, 3)
    }

In [ ]:
samples = {
    "ch2.pdf": [9, 17, 65],
    "scanned.pdf": [1, 2, 3]
}

for filename, page_numbers in samples.items():
    with pymupdf.open(filename) as doc:
        for page_number in page_numbers:
            features = inspect_page(doc[page_number - 1])
            print(filename, page_number, features)

ch2.pdf 9 {'native_words': 148, 'image_count': 0, 'largest_image_fraction': 0.0}
ch2.pdf 17 {'native_words': 167, 'image_count': 0, 'largest_image_fraction': 0.0}
ch2.pdf 65 {'native_words': 39, 'image_count': 2, 'largest_image_fraction': 0.807}
scanned.pdf 1 {'native_words': 2, 'image_count': 1, 'largest_image_fraction': 1.0}
scanned.pdf 2 {'native_words': 0, 'image_count': 1, 'largest_image_fraction': 1.0}
scanned.pdf 3 {'native_words': 0, 'image_count': 1, 'largest_image_fraction': 1.0}


In [ ]:
def select_method(features):
    words = features["native_words"]
    coverage = features["largest_image_fraction"]

    if coverage >= 0.90:
        if words == 0:
            return "OCR"
        elif words < 20:
            return "HYBRID"
        else:
            return "REVIEW"

    if words > 0:
        return "NATIVE"

    return "REVIEW"

In [ ]:
for filename, page_numbers in samples.items():
    with pymupdf.open(filename) as doc:
        for page_number in page_numbers:
            features = inspect_page(doc[page_number - 1])

            method = select_method(features)

            print(f"{filename}, page {page_number}: {method}")

ch2.pdf, page 9: NATIVE
ch2.pdf, page 17: NATIVE
ch2.pdf, page 65: NATIVE
scanned.pdf, page 1: HYBRID
scanned.pdf, page 2: OCR
scanned.pdf, page 3: OCR


### Configure OCR language data

`extract_page()` defaults to the local `TESSDATA_DIR` when no `tessdata` argument is supplied. It checks the requested language models before running OCR; pass `tessdata="/another/directory"` to override the default. The English model is available at `tessdata/eng.traineddata`. This avoids relying on a system Tesseract installation to locate the models.

`select_method()` is a heuristic: choosing `NATIVE` does not guarantee that text inside images is recovered. For example, page 65 contains large images but falls below the 90% coverage threshold.

In [ ]:
def extract_page(page, method, language="eng", tessdata=None):
    if method == "NATIVE":
        return page.get_text("text")

    elif method in ("OCR", "HYBRID"):
        # Use the project's language models unless a custom directory is given.
        tessdata_dir = TESSDATA_DIR if tessdata is None else Path(tessdata)
        for lang in language.split("+"):
            model_path = tessdata_dir / f"{lang}.traineddata"
            if not model_path.is_file():
                raise FileNotFoundError(f"OCR language model not found: {model_path}")

        tp = page.get_textpage_ocr(
            language=language,
            dpi=200,
            full=(method == "OCR"),
            tessdata=str(tessdata_dir)
        )
        return page.get_text("text", textpage=tp)

    elif method == "REVIEW":
        return None

    else:
        raise ValueError(f"Unknown extraction method: {method}")

In [ ]:
samples = {
    "ch2.pdf": [9, 65],
    "scanned.pdf": [1, 2]
}

for filename, page_numbers in samples.items():
    with pymupdf.open(filename) as doc:
        for page_number in page_numbers:
            page = doc[page_number - 1]

            # Step 1: Inspect
            features = inspect_page(page)

            # Step 2: Select method
            method = select_method(features)

            # Step 3: Extract
            text = extract_page(page, method, tessdata=str(TESSDATA_DIR))

            print(f"\n{filename} | Page {page_number}")
            print("Method:", method)

            if text is None:
                print("Needs review")
            else:
                print("Characters:", len(text))
                print("Preview:", repr(text[:200]))


ch2.pdf | Page 9
Method: NATIVE
Characters: 845
Preview: 'Why the support of the prior matters. II\n• In the dominated case this is Bayes’ rule: π(A|x) =\n1\nm(x)\nR\nA f(x|θ)π(θ) dθ = 0\nas soon as π(A) =\nR\nA π(θ) dθ = 0.\n• Cromwell’s rule (Lindley 1985): never g'

ch2.pdf | Page 65
Method: NATIVE
Characters: 244
Preview: 'The ML-II approach: Random Effects example\nIntegrated likelihood L(τ 2) and profile likelihood ˆL(τ 2) (Berger, Liseo and Wolpert\n1999, Figure 1). The integrated likelihood has a well defined maximum '

scanned.pdf | Page 1
Method: HYBRID
Characters: 2870
Preview: 'SAMPLE LETTER\nse STA.\n2 re \\ i}\nUNITED STATES ENVIRONMENTAL PROTECTION AGENCY\n3SZ &\nWASHINGTON, D.C. 20460\n&,\neo\nAL pro\nPE.\ny\n2016\nOFFICE OF\nWATER\nDear Commissioner:\nThere is no higher priority for th'

scanned.pdf | Page 2
Method: OCR
Characters: 3373
Preview: 'othe materials inventory that systems were required to complete under the LCR, including\nthe locations of lead service lin

In [ ]:
with pymupdf.open("scanned.pdf") as doc:
    page = doc[0]

    native_text = page.get_text("text")
    hybrid_text = extract_page(page, "HYBRID")

    print("Native words:", len(native_text.split()))
    print("Hybrid words:", len(hybrid_text.split()))

    print("\nExisting heading preserved:",
          "SAMPLE LETTER" in hybrid_text)

Native words: 2
Hybrid words: 458

Existing heading preserved: True
